# Lord of the Rings NLP Refactor
This notebook compares classic topic models and embedding-based topic modeling for Tolkien's trilogy.
It uses chapter-level documents and includes coherence scoring plus timeline visualization.


In [ ]:
from pathlib import Path
import lotr_nlp

lotr_nlp.ensure_nltk_resources()
BOOK_DIR = Path("the_lord_of_the_rings_text")

book_texts = lotr_nlp.load_book_texts(BOOK_DIR)
print("Loaded books:")
for book_name, _ in book_texts:
    print(f"- {book_name}")

chapter_docs = lotr_nlp.build_book_chapter_documents(
    BOOK_DIR,
    preserve_case=False,
    extra_stopwords=["said", "come", "came"],
)
print(f"Built {len(chapter_docs)} chapter documents across the trilogy")


## Chapter-level LDA
Fit LDA on chapter documents, show top topic terms, compute coherence, and plot topic prevalence across the trilogy timeline.


In [ ]:
documents = [chapter['document'] for chapter in chapter_docs]
tokenized_documents = [chapter['tokens'] for chapter in chapter_docs]

lda_matrix, lda_vectorizer = lotr_nlp.build_document_term_matrix(documents)
lda_model = lotr_nlp.fit_lda(lda_matrix, n_components=10)
lda_terms = lotr_nlp.get_topic_term_lists(lda_model, lda_vectorizer.get_feature_names_out(), 15)

print("LDA topic terms:")
lotr_nlp.print_top_words(lda_model, lda_vectorizer.get_feature_names_out(), n_top_words=15)
print(f"\nLDA coherence (c_v): {lotr_nlp.compute_coherence(lda_terms, tokenized_documents):.4f}")

lda_fig = lotr_nlp.plot_topic_trends(
    lda_model.transform(lda_matrix),
    chapter_docs,
    top_n=5,
    title="Chapter-level LDA Topic Trends",
)
lda_fig


## TF-IDF + NMF
Fit NMF on TF-IDF chapter documents, print topic terms, compute coherence, and plot topic series.


In [ ]:
tfidf_matrix, tfidf_vectorizer = lotr_nlp.build_tfidf_matrix(documents)
nmf_model = lotr_nlp.fit_nmf(tfidf_matrix, n_components=10)
nmf_terms = lotr_nlp.get_topic_term_lists(nmf_model, tfidf_vectorizer.get_feature_names_out(), 15)

print("NMF topic terms:")
lotr_nlp.print_top_words(nmf_model, tfidf_vectorizer.get_feature_names_out(), n_top_words=15)
print(f"\nNMF coherence (c_v): {lotr_nlp.compute_coherence(nmf_terms, tokenized_documents):.4f}")

nmf_fig = lotr_nlp.plot_topic_trends(
    nmf_model.transform(tfidf_matrix),
    chapter_docs,
    top_n=5,
    title="TF-IDF + NMF Topic Trends",
)
nmf_fig


## Embedding-based Topic Modeling
Cluster chapter embeddings with KMeans, extract top cluster terms, and score coherence.


In [ ]:
labels, embeddings, embedding_model = lotr_nlp.fit_embedding_topic_model(documents, n_clusters=10)
cluster_topics = lotr_nlp.get_cluster_topic_terms(documents, labels, n_top_words=15)

print("Embedding-based clusters:")
for label, terms in cluster_topics:
    print(f"Cluster {label}: {' '.join(terms)}")

print(f"\nEmbedding coherence (c_v): {lotr_nlp.compute_coherence([terms for _, terms in cluster_topics], tokenized_documents):.4f}")


## Timeline Summary
The two timeline plots show how the strongest topics evolve across the trilogy, using chapter order as the narrative timeline.


In [ ]:
print("LDA timeline plot:")
lda_fig

print("NMF timeline plot:")
nmf_fig


## Notes
- The chapter-level models use the three individual trilogy files rather than a single monolithic text.
- Embedding-based clustering is a semantic complement to the bag-of-words methods.
- Coherence scores help compare model interpretability across approaches.
